# 04 可解释性分析
从开发队列中挑选预测正确和错误的代表病例，查看四层注意力权重和Grad-CAM热图。

每位患者使用其所在留出折对应的模型。患者未参与该模型的梯度训练，但当前02会用留出折选择最佳轮，所以这些图属于开发阶段解释。热图表示模型输出对影像特征的响应，不等同于病灶分割或因果解释。


## 选择要解释的实验


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "config.json").exists() and (ROOT / "1117" / "config.json").exists():
    ROOT = ROOT / "1117"
cfg = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))

def path_from_config(key):
    value = Path(cfg[key]).expanduser()
    return value if value.is_absolute() else ROOT / value

CACHE = path_from_config("cache_path")
COHORT = path_from_config("cohort_path")
RESULTS = path_from_config("results_dir")
SEED = 20260922
CLINICAL = ["age", "education", "nihss", "hcy", "ua"]
SEQUENCES = ["t1", "t2", "dwi"]
IMAGE_FILES = ["t1_flair.nii.gz", "t2.nii.gz", "dwi_b1000.nii.gz"]
print("患者清单：", COHORT)
print("结果目录：", RESULTS)


In [ ]:
def read_cases():
    cases = pd.read_csv(COHORT, dtype={"mr": str})
    if not cases.mr.is_unique or not set(cases.split) <= {"development", "validation"}:
        raise ValueError("患者编号重复或分组有误，请检查cohort.csv。")
    if not set(cases.label) <= {0, 1} or cases.mr.isna().any():
        raise ValueError("患者编号缺失或结局不是0/1。")
    return cases


def read_classification_data():
    cases = read_cases()
    with np.load(CACHE, allow_pickle=False) as data:
        cache_mrs = data["mrs"].astype(str).tolist()
        if len(set(cache_mrs)) != len(cache_mrs) or set(cache_mrs) != set(cases.mr):
            raise ValueError("缓存与患者清单的MR编号不一致。")
        lookup = {mr: i for i, mr in enumerate(cache_mrs)}
        order = [lookup[mr] for mr in cases.mr]
        if not np.array_equal(data["labels"][order], cases.label) or not np.array_equal(data["max_layers"][order], cases.max_layer):
            raise ValueError("患者标签或取层与缓存不一致。")
        images = data["images"][order].astype(np.float32)
        masks = data["masks"][order].astype(np.uint8)
    if images.ndim != 5 or images.shape[1:3] != (4, 3) or masks.shape != images.shape or min(images.shape[-2:]) < 32:
        raise ValueError("分类输入应为[N,4,3,H,W]，影像与mask尺寸须一致，H和W至少为32。")
    if not np.isfinite(images).all() or not np.isin(masks, [0, 1]).all():
        raise ValueError("影像存在无效值或mask不是0/1。")
    return cases, images, masks


In [ ]:
cases, images, masks = read_classification_data()
MODEL_DIR = RESULTS / "resnet"
RUN_EXPLAIN = False
# 如需指定病例，填写开发队列MR编号；留空则每类选一例。
CASE_MRS = []


## 读取模型，计算四层权重及 Grad-CAM
模型从PT权重读取输入标准化设置；与02一样，只在ResNet入口处理影像，不改变mask。显示的灰度图仍使用缓存中的0～1强度，热图放回同样大小的像素网格。

缓存尺寸需与这份权重训练时一致。换成原尺寸缓存后，先完成02重训，再运行本节，不能混用旧224权重。


In [ ]:
import random
import torch
from torch import nn
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.transforms.functional import affine
from torchvision.transforms import InterpolationMode

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("计算设备：", DEVICE)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class GatedAttention(nn.Module):
    def __init__(self, features, hidden=128):
        super().__init__()
        self.value = nn.Linear(features, hidden)
        self.gate = nn.Linear(features, hidden)
        self.score = nn.Linear(hidden, 1)

    def forward(self, x):
        return self.score(torch.tanh(self.value(x)) * torch.sigmoid(self.gate(x)))


class MaskEncoder(nn.Module):
    def __init__(self, features=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1, bias=False), nn.GroupNorm(8, 32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1, bias=False), nn.GroupNorm(8, 64), nn.ReLU(),
            nn.Conv2d(64, 128, 3, 2, 1, bias=False), nn.GroupNorm(8, 128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, features),
            nn.LayerNorm(features), nn.GELU(), nn.Dropout(.3))

    def forward(self, x):
        return self.net(x.float())


class ResNetClassifier(nn.Module):
    def __init__(self, image_dim=256, mask_dim=64, attention_dim=128,
                 use_mask=True, use_attention=True, pretrained=False, normalize_input=False):
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        self.image_features = nn.Sequential(*list(backbone.children())[:-1])
        self.image_features.requires_grad_(False)
        self.image_features.eval()
        self.normalize_input = normalize_input
        self.register_buffer("input_mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("input_std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1), persistent=False)
        self.image_projection = nn.Sequential(nn.Linear(2048, image_dim), nn.LayerNorm(image_dim), nn.GELU(), nn.Dropout(.3))
        self.mask_features = MaskEncoder(mask_dim) if use_mask else None
        width = image_dim + (mask_dim if use_mask else 0)
        self.attention = GatedAttention(width, attention_dim) if use_attention else None
        self.head = nn.Sequential(nn.LayerNorm(width), nn.Dropout(.4), nn.Linear(width, 1))

    def train(self, mode=True):
        super().train(mode)
        # 后续模块照常训练，固定主干及其BatchNorm统计量。
        self.image_features.eval()
        return self

    def forward(self, images, masks):
        b, slices, channels, h, w = images.shape
        image_input = images.reshape(b * slices, channels, h, w)
        if self.normalize_input:
            image_input = (image_input - self.input_mean) / self.input_std
        features = self.image_projection(self.image_features(image_input).flatten(1))
        features = features.reshape(b, slices, -1)
        if self.mask_features is not None:
            lesion = self.mask_features(masks.reshape(b * slices, 3, h, w)).reshape(b, slices, -1)
            features = torch.cat([features, lesion], dim=-1)
        if self.attention is None:
            pooled = features.mean(dim=1)
        else:
            weights = self.attention(features).softmax(dim=1)
            pooled = (features * weights).sum(dim=1)
        return self.head(pooled).squeeze(-1)


def sequence_input(images, sequence="all"):
    if sequence == "all":
        return images
    channel = SEQUENCES.index(sequence)
    return images[:, :, [channel, channel, channel]]


@torch.no_grad()
def predict(model, images, masks, batch_size=16):
    model.eval()
    probabilities = []
    for start in range(0, len(images), batch_size):
        x = torch.as_tensor(images[start:start + batch_size], dtype=torch.float32, device=DEVICE)
        m = torch.as_tensor(masks[start:start + batch_size], dtype=torch.float32, device=DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            logits = model(x, m)
        probabilities.append(logits.float().sigmoid().cpu().numpy())
    return np.concatenate(probabilities)


def load_classifier(file):
    saved = torch.load(file, map_location="cpu", weights_only=True)
    model = ResNetClassifier(**saved["model_args"], pretrained=False).to(DEVICE)
    model.load_state_dict(saved["state_dict"])
    return model.eval(), saved


In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F


def select_cases(cases, model_dir):
    oof = pd.read_csv(model_dir / "development_oof.csv", dtype={"mr": str})
    threshold = float(pd.read_csv(model_dir / "development_metrics.csv").iloc[0].threshold)
    table = cases.query("split == 'development'").merge(oof[["mr", "probability"]], on="mr", validate="one_to_one")
    positive = table.probability >= threshold
    table["category"] = np.select([(table.label == 1) & positive, (table.label == 0) & ~positive,
                                   (table.label == 0) & positive], ["TP", "TN", "FP"], default="FN")
    return table.sort_values("mr").groupby("category", sort=True).head(1)


def grad_cam(model, images, masks):
    captured = {}
    def feature_hook(module, inputs, output):
        captured["features"] = output
        output.retain_grad()
    def attention_hook(module, inputs, output):
        captured["attention"] = output
    hooks = [model.image_features[7].register_forward_hook(feature_hook)]
    if model.attention is not None:
        hooks.append(model.attention.register_forward_hook(attention_hook))
    try:
        model.eval()
        model.zero_grad(set_to_none=True)
        logits = model(images.requires_grad_(True), masks)
        logits.sum().backward()
        features = captured["features"]
        weights = features.grad.mean(dim=(2, 3), keepdim=True)
        heatmap = torch.relu((features * weights).sum(dim=1, keepdim=True))
        heatmap = F.interpolate(heatmap, size=images.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
        heatmap = heatmap / heatmap.amax(dim=(1, 2), keepdim=True).clamp_min(1e-8)
        attention = captured["attention"].softmax(dim=1)[0, :, 0] if "attention" in captured else torch.full((4,), .25, device=DEVICE)
        return heatmap.detach().cpu().numpy(), attention.detach().cpu().numpy(), float(logits.sigmoid().item())
    finally:
        for hook in hooks:
            hook.remove()
        model.zero_grad(set_to_none=True)


def explain_cases(cases, images, masks, selected, model_dir):
    output = model_dir / "explain"
    output.mkdir(parents=True, exist_ok=True)
    lookup = {mr: i for i, mr in enumerate(cases.mr)}
    rows = []
    for _, patient in selected.iterrows():
        model, saved = load_classifier(model_dir / f"fold_{int(patient.development_fold)}.pt")
        if patient.mr in saved["train_mrs"]:
            raise ValueError("该患者出现在当前折训练集中，请核对折号。")
        i = lookup[patient.mr]
        original = images[i:i + 1]
        expected_size = tuple(saved.get("input_size", [224, 224]))
        if tuple(original.shape[-2:]) != expected_size:
            raise ValueError(f"该权重使用{expected_size}输入，当前缓存为{original.shape[-2:]}。请先用当前缓存完成02训练，再运行04。")
        x = torch.from_numpy(sequence_input(original, saved["sequence"])).to(DEVICE)
        m = torch.from_numpy(masks[i:i + 1]).to(DEVICE)
        cam, attention, probability = grad_cam(model, x, m)
        fig, axes = plt.subplots(4, 4, figsize=(11, 11))
        for s in range(4):
            for channel, title in enumerate(("T1-FLAIR", "T2", "DWI")):
                axes[channel, s].imshow(original[0, s, channel], cmap="gray", vmin=0, vmax=1)
                axes[channel, s].set_title(f"{title} | slice {s + 1}")
            axes[3, s].imshow(original[0, s, 0], cmap="gray", vmin=0, vmax=1)
            axes[3, s].imshow(cam[s], cmap="jet", alpha=.4)
            axes[3, s].set_title(f"CAM | weight={attention[s]:.3f}")
        for ax in axes.flat:
            ax.axis("off")
        fig.suptitle(f"{patient.mr} | {patient.category} | y={patient.label} | p={probability:.3f}")
        fig.tight_layout()
        fig.savefig(output / f"{patient.mr}.png", dpi=150)
        plt.close(fig)
        rows.append({"mr": patient.mr, "category": patient.category, "label": int(patient.label), "probability": probability,
                     **{f"slice_{s + 1}_weight": float(attention[s]) for s in range(4)}})
        del model
    result = pd.DataFrame(rows)
    result.to_csv(output / "cases.csv", index=False)
    return result


## 选择病例
默认在 TP、TN、FP、FN 四类中按 MR 编号各取一例，并非按热图好看程度选择。预览和绘图始终使用上方同一个 `MODEL_DIR`。

Grad-CAM 针对“结局为 1”的输出计算，因此阴性病例也显示支持阳性输出的区域。


In [ ]:
if (MODEL_DIR / "development_oof.csv").exists():
    selected = select_cases(cases, MODEL_DIR)
    if CASE_MRS:
        oof = pd.read_csv(MODEL_DIR / "development_oof.csv", dtype={"mr": str})
        selected = cases.query("split == 'development'").merge(oof[["mr", "probability"]], on="mr", validate="one_to_one")
        selected = selected.loc[selected.mr.isin(CASE_MRS)].copy()
        if set(selected.mr) != set(CASE_MRS):
            raise ValueError("指定病例必须都属于开发队列。")
        threshold = float(pd.read_csv(MODEL_DIR / "development_metrics.csv").iloc[0].threshold)
        positive = selected.probability >= threshold
        selected["category"] = np.select([(selected.label == 1) & positive, (selected.label == 0) & ~positive,
                                          (selected.label == 0) & positive], ["TP", "TN", "FP"], default="FN")
    display(selected[["mr", "label", "probability", "category", "development_fold"]])
    if RUN_EXPLAIN:
        display(explain_cases(cases, images, masks, selected, MODEL_DIR))
else:
    print("先完成所选实验的五折训练。")


结果位于所选实验下的 `explain` 文件夹：每例一张 PNG，`cases.csv` 记录概率和四层权重。改变实验目录即可解释某个消融模型；去注意力模型的四层权重显示为相同的 0.25。
